# MCP — extracted from `01_agentic_patterns.ipynb`

These are the Part 5 cells, verbatim, saved when MCP was pulled out of the
framework-zoo notebook. This is a holding file, not a finished notebook.

To run the code cells standalone you need `import inspect, json` and the
`calculator` function from Part 1 of notebook 01.


---
# Part 5 — MCP

> **Control flow: nobody.** MCP is plumbing, not an architecture.

### 5.1 MPC is not MCP

Two three-letter acronyms, one letter apart, both in this notebook. Sorry.

| **MPC** — Model Predictive Control | **MCP** — Model Context Protocol |
|---|---|
| A *control strategy* | A *wire protocol* |
| From process control, 1970s | Introduced by Anthropic, 2024 |
| "How do I decide what to do next?" | "How does my agent reach a tool that lives somewhere else?" |
| Changes your agent's reasoning | Changes your agent's plumbing |

They are orthogonal. You can — and often should — run an MPC-style agent whose tools are served
over MCP.

### 5.2 The problem it solves

Every tool so far has been a Python function in the same process as the loop. That does not
survive contact with a real simulation stack.

Your solver is a compiled binary, on a cluster, behind a scheduler, written by someone else, in
Fortran, in 1994, licensed per seat.

```
ReAct agent   ─┐                    ┌─▶  mesh server
LangGraph app ─┼──▶   [  MCP  ]  ───┼─▶  solver server
Claude / IDE  ─┘                    └─▶  CAD server
```

Write your solver's tool surface **once**; any MCP-speaking client can drive it — your notebook
today, a colleague's LangGraph pipeline tomorrow, a desktop assistant after that.

### 5.3 The syntax

Server side, you decorate the functions you want to expose. The type hints and docstrings
*become* the JSON schema.

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("fem-tools")

@mcp.tool()
def solve_poisson(mesh_id: int) -> dict:
    """Solve -Laplacian(u) = 1 with u = 0 on the boundary."""
    u, basis = solve_poisson_on(MESH_REGISTRY[mesh_id])
    return {"dofs": int(basis.N), "H1_error": h1_error(...)}

@mcp.tool()
def refine_by_threshold(mesh_id: int, theta: float = 0.5) -> dict:
    """Doerfler-mark and refine. theta in (0,1)."""
    ...

mcp.run()          # now speaks MCP over stdio or HTTP
```

Client side, you point at the server and the tools show up. Nothing about *what a tool is*
changed — only **who can reach it**.

### 5.4 The same thing in 15 lines

The decorator is doing one interesting thing: turning a signature into a schema. Here it is,
so it stops looking like magic.

In [ ]:
def schema_from_function(fn):
    """Type hints + docstring -> JSON schema. This is the useful half of @mcp.tool()."""
    tmap = {int: "integer", float: "number", str: "string", bool: "boolean"}
    props, required = {}, []
    for name, p in inspect.signature(fn).parameters.items():
        props[name] = {"type": tmap.get(p.annotation, "string")}
        if p.default is inspect.Parameter.empty:
            required.append(name)
        else:
            props[name]["description"] = f"default: {p.default!r}"
    return {"name": fn.__name__,
            "description": (fn.__doc__ or "").strip().split("\n")[0],
            "parameters": {"type": "object", "properties": props, "required": required}}


def refine_by_threshold(mesh_id: int, theta: float = 0.5) -> dict:
    """Doerfler-mark and refine the given mesh. theta in (0,1)."""
    return {}

print(json.dumps(schema_from_function(refine_by_threshold), indent=2))

In [ ]:
# It works on the tools we already wrote, too -- no rewriting needed.
print(json.dumps(schema_from_function(calculator), indent=2))

**Compare that against `CALC_PARAMS` in Part 1.** The generated schema lost the
`"enum": ["add","sub","mul","div"]` — a plain `str` annotation cannot express it.

That is not a nitpick. Part 1 argued that every value you forbid in the schema is a failure mode
you never have to debug; naive schema generation just gave those back. Real MCP servers recover
them with `Literal["add","sub","mul","div"]` or a Pydantic model:

```python
from typing import Literal

@mcp.tool()
def calculator(op: Literal["add","sub","mul","div"], a: float, b: float) -> dict:
    """Perform one arithmetic operation."""
```

**The general lesson:** convenience layers that infer schemas are inferring from whatever your
type hints happen to say. Check what they produced before you trust it — the schema is the
contract with the model, and a weaker contract fails quietly.

### 5.5 When to reach for MCP

**Use it when** your tools live in another process, language, or machine — or when other people
need to reuse them. Exposing a solver over MCP is how a one-off notebook demo becomes shared
infrastructure.

**Skip it when** everything is one Python process and one consumer. It is a serialization
boundary you would be adding for its own sake.

**Note it is compatible with all four architectures above.** MCP answers "where do tools live,"
which is a different question from "who decides what to do next."